# 6.2.4 Direct Alignment Methods (DPO)

Demonstrate the DPO loss formula and compare DPO vs SFT loss curves on synthetic preference data.
DPO loss: $-\log \sigma(\beta((\log \pi(y_w) - \log \pi_{ref}(y_w)) - (\log \pi(y_l) - \log \pi_{ref}(y_l))))$

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)
beta = 0.1

def dpo_loss(log_pi_w, log_pi_l, log_ref_w, log_ref_l, beta=0.1):
    ratio = beta * ((log_pi_w - log_ref_w) - (log_pi_l - log_ref_l))
    return -np.log(1.0 / (1.0 + np.exp(-ratio)))

n = 500
log_ref_w = rng.normal(-2, 0.5, n)
log_ref_l = rng.normal(-3, 0.5, n)

baseline_dpo = dpo_loss(log_ref_w, log_ref_l, log_ref_w, log_ref_l).mean()
print(f'DPO loss (policy = ref): {baseline_dpo:.4f}  (should be log(2) ≈ {np.log(2):.4f}')

In [ ]:
steps = np.arange(1, 101)
dpo_losses, sft_losses = [], []
for s in steps:
    a = s / 100
    log_pi_w = log_ref_w + a * 0.5
    log_pi_l = log_ref_l - a * 0.3
    dpo_losses.append(dpo_loss(log_pi_w, log_pi_l, log_ref_w, log_ref_l).mean())
    sft_losses.append(-log_pi_w.mean() + rng.normal(0, 0.02))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(steps, dpo_losses, color='steelblue', lw=2, label='DPO')
axes[0].plot(steps, sft_losses, color='tomato', lw=2, label='SFT NLL')
axes[0].set(xlabel='Training Steps', ylabel='Loss', title='DPO vs SFT Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

margins = np.linspace(-3, 3, 200)
axes[1].plot(margins, -np.log(1/(1+np.exp(-beta*margins))), color='darkorange', lw=2)
axes[1].set(xlabel='β(r_w − r_l)', ylabel='DPO Loss', title='DPO Loss vs Reward Margin')
axes[1].axvline(0, color='gray', linestyle='--'); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/dpo_demo.png')
print('Saved dpo_demo.png')